# Campaign-setup agent — walkthrough

This notebook runs the first YieldAgent vertical slice end-to-end against a **Meta test ad account**: parse a markdown brief, plan a draft `Campaign`, approve at the human gate, publish to Meta as `PAUSED` objects.

It also renders the LangGraph topology as a PNG and saves it under `docs/img/` for reuse in the docs and README.

## Requirements

- `pip install -e ".[meta,agent]"` from the repo root (and `pip install jupyter`).
- A `.env` at the repo root with `ANTHROPIC_API_KEY`, `META_ACCESS_TOKEN`, `META_AD_ACCOUNT_ID`, `META_PAGE_ID`. See `.env.example`.
- The configured Meta ad account must be a **test ad account**. The Meta MCP server refuses live accounts unless `YIELDAGENT_ALLOW_LIVE=1` is set, and you should not set that here.

> The publish step will create real (paused) objects in your Meta test account. Nothing goes live.

## 1. Load environment

Reads `../.env` so the Meta MCP subprocess and `ChatAnthropic` both pick up credentials. Falls back gracefully if `python-dotenv` isn't installed.

In [ ]:
import os
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ENV_PATH = REPO_ROOT / ".env"

def _load_env(path: Path) -> int:
    if not path.exists():
        return 0
    loaded = 0
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value
            loaded += 1
    return loaded

loaded = _load_env(ENV_PATH)
print(f"Loaded {loaded} variable(s) from {ENV_PATH}")

required = ["ANTHROPIC_API_KEY", "META_ACCESS_TOKEN", "META_AD_ACCOUNT_ID", "META_PAGE_ID"]
missing = [k for k in required if not os.environ.get(k)]
assert not missing, f"Missing required env vars: {missing}"
assert os.environ.get("YIELDAGENT_ALLOW_LIVE") != "1", (
    "YIELDAGENT_ALLOW_LIVE=1 is set. Unset it before running this notebook — "
    "the notebook is intended for test ad accounts only."
)
print("Env looks good. Ad account:", os.environ["META_AD_ACCOUNT_ID"])

## 2. Build the graph

`build_graph()` wires the four nodes and spawns the Meta MCP server as a stdio subprocess on the first tool call. Each invocation needs a `thread_id` so the checkpointer can resume the run after the `human_gate` interrupt.

In [ ]:
from uuid import uuid4

from langgraph.types import Command

from yieldagent.agents.campaign_setup import build_graph

graph = build_graph()
config = {"configurable": {"thread_id": str(uuid4())}}
config

## 3. Visualize the graph

Render the LangGraph topology with Mermaid and write the PNG to `docs/img/campaign_setup_graph.png` so the README and docs can embed it without re-running the notebook.

In [ ]:
from IPython.display import Image, display

png_bytes = graph.get_graph().draw_mermaid_png()

png_path = REPO_ROOT / "docs" / "img" / "campaign_setup_graph.png"
png_path.parent.mkdir(parents=True, exist_ok=True)
png_path.write_bytes(png_bytes)
print(f"Wrote {len(png_bytes)} bytes to {png_path.relative_to(REPO_ROOT)}")

display(Image(png_bytes))

## 4. Case — Glow Roast "Midnight Brew" launch

Load the reference brief and run the agent. The graph will execute `parse_brief` and `plan_campaign`, then pause at `human_gate` via LangGraph's `interrupt()`. The first `ainvoke` returns the state with an `__interrupt__` key — that's the cue to review the plan before resuming.

In [ ]:
brief_path = REPO_ROOT / "briefs" / "example_brief.md"
brief_text = brief_path.read_text()
print(brief_text[:400], "...\n")

paused_state = await graph.ainvoke({"brief_text": brief_text}, config=config)
assert "__interrupt__" in paused_state, "Expected the graph to pause at human_gate"
print("Paused at human_gate. Audit trail so far:")
for entry in paused_state.get("audit", []):
    print(f"  [{entry['node']}] {entry['summary']}")

### Review the planned draft

`plan_campaign` always forces `status='draft'` and the Meta client always creates `PAUSED` objects — so what you see below is what would be created.

In [ ]:
import json

campaign = paused_state["campaign"]
print(json.dumps(campaign.model_dump(mode="json"), indent=2))

## 5. Approve at the gate and publish

Resume the graph with `Command(resume={"approved": True, ...})`. The conditional edge routes to `publish_draft`, which calls the Meta MCP `publish_draft_campaign` tool and returns the created object IDs.

In [ ]:
final = await graph.ainvoke(
    Command(resume={"approved": True, "reason": "approved in notebook walkthrough"}),
    config=config,
)

print("Publish result:")
print(json.dumps(final.get("publish_result", {}), indent=2))

## 6. Full audit trail

Pillar 6 in action: every node appended an `AuditEntry`. This is the immutable record of what the agent did and why — what a human (or another agent) would inspect after the fact.

In [ ]:
for entry in final.get("audit", []):
    print(f"[{entry['node']}] {entry['summary']}")
    for k, v in entry["detail"].items():
        print(f"    {k}: {v}")

## Cleanup

The draft campaign, ad sets, and ads now exist in your Meta test account in `PAUSED` state. Delete them from Meta Business Manager when you're done — the agent intentionally has no delete tool, so cleanup is a human action.